# Exploration des Survey Property Maps — DP2 (USDF) — Représentation HEALPix native

Ce notebook est dérivé de `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb`.

Il adopte une **approche alternative de visualisation** inspirée de `read_healsparse.ipynb` :
au lieu d'afficher les images PNG pré-générées par le pipeline ou d'utiliser `skyproj`,
on convertit chaque `HealSparseMap` en carte HEALPix dense (`healsparse → healpy`)
et on l'affiche avec `hp.mollview` / `hp.gnomview`, ce qui permet :

- un contrôle fin de la projection (Mollweide, gnomonic, orthographic…),
- un repère de grille (graticule) avec labels de coordonnées,
- une superposition du plan Galactique,
- une normalisation couleur adaptative (linéaire, LogNorm, SymLogNorm).

**Table des matières**

1. Imports
2. Configuration Butler
3. Définitions des fonctions utilitaires
4. Initialisation du Butler
5. Explorer les collections disponibles dans `dp2_prep`
6. Recherche des dataset types de Survey Property Maps
7. Séparation objets HealSparseMap / images PNG
8. Disponibilité des HealSparse maps dans la collection cible
9A. Flags de configuration de l'affichage
9B. Fonction helper — plan Galactique
9C. Affichage HEALPix — grille 2×3 bandes (healpy)
9D. Fallback PNG — grille 2×3 bandes
10. Valeur de la carte au champ de référence
11. Statistiques dans une région autour du champ de référence
12. Visualisation locale (pcolormesh)
13. Vue tout-ciel healpy — boucle sur toutes les cartes disponibles
14. Notes et prochaines étapes

| Champ | Valeur |
|---|---|
| **Auteur** | Sylvie Dagoret-Campagne |
| **Date de création** | 2026-03-12 |
| **Environnement** | USDF RSP — kernel `LSST` (`lsst_distrib` stack) |
| **Notebook de référence DP2** | `2026-03-11_ExploreDP2_SurveyPropertyMaps.ipynb` |
| **Notebook healsparse** | `read_healsparse.ipynb` |
| **DP2 DRP Campaigns** | https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs |

| Dataset type | Description | Unité |
|---|---|---|
| `deepCoadd_exposure_time_consolidated_map_sum` | Temps de pose total accumulé par position de ciel | **seconde** |
| `deepCoadd_epoch_consolidated_map_min` | Époque de la première observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_max` | Époque de la dernière observation | **MJD** |
| `deepCoadd_epoch_consolidated_map_mean` | Époque moyenne de toutes les observations contributives | **MJD** |
| `deepCoadd_psf_size_consolidated_map_weighted_mean` | Moyenne pondérée de la largeur caractéristique de la PSF (rayon déterminant) | **pixel** |
| `deepCoadd_psf_e1_consolidated_map_weighted_mean` | Moyenne pondérée de la composante d'ellipticité PSF e1 | sans dimension |
| `deepCoadd_psf_e2_consolidated_map_weighted_mean` | Moyenne pondérée de la composante d'ellipticité PSF e2 | sans dimension |
| `deepCoadd_psf_maglim_consolidated_map_weighted_mean` | Moyenne pondérée de la limite en magnitude 5σ de la PSF | **mag AB** |
| `deepCoadd_sky_background_consolidated_map_weighted_mean` | Moyenne pondérée du niveau de fond de ciel | **nJy** |
| `deepCoadd_sky_noise_consolidated_map_weighted_mean` | Moyenne pondérée de l'écart-type du fond de ciel | **nJy** |

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.lines as mlines
from matplotlib.colors import LogNorm, SymLogNorm, Normalize
import pandas as pd
import os
import re
from pathlib import Path
from IPython.display import display

# LSST middleware
from lsst.daf.butler import Butler

# Astropy — coordonnées Galactiques pour le plan Galactique
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u

# healpy — visualisation HEALPix native (Mollweide, gnomview, etc.)
import healpy as hp

# healsparse — lecture des HealSparseMap, conversion vers HEALPix dense
try:
    import healsparse
    HAS_HEALSPARSE = True
    print('healsparse available')
except ImportError:
    HAS_HEALSPARSE = False
    print('healsparse not available — HealSparse sections will be skipped')

# skyproj (optionnel — conservé pour compatibilité)
try:
    import skyproj
    HAS_SKYPROJ = True
    print('skyproj available')
except ImportError:
    HAS_SKYPROJ = False
    print('skyproj not available')

print('All imports done.')

## 2. Configuration Butler

In [ ]:
# ── Butler repository au USDF ────────────────────────────────────────────────
REPO = 'dp2_prep'

# ── Collection cible pour les survey property maps ───────────────────────────
COLLECTION = 'LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545'

# ── Skymap ────────────────────────────────────────────────────────────────────
SKYMAP_NAME = 'lsst_cells_v2'
INSTRUMENT  = 'LSSTCam'

# ── Bandes photométriques ─────────────────────────────────────────────────────
BANDS    = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_REF = 'r'
band_order = {b: i for i, b in enumerate(BANDS)}

# ── Coordonnées des champs de référence ──────────────────────────────────────
RA_ECDFS  = 53.13
DEC_ECDFS = -28.10

RA_COSMOS  = 150.12
DEC_COSMOS = 2.21

FIELD   = 'COSMOS'
RA_CEN  = RA_COSMOS
DEC_CEN = DEC_COSMOS

# ── Résolution HEALPix pour la conversion HealSparse → healpy ────────────────
# nside=32  : vue tout-ciel rapide (~1.8°/pixel)
# nside=64  : bon compromis résolution/mémoire (~0.9°/pixel)
# nside=128 : haute résolution (~0.46°/pixel) — plus lent
NSIDE_DISPLAY = 64

# ── Répertoires de sortie ─────────────────────────────────────────────────────
NB_TAG   = 'DP2_SPMAP_HEALPIX'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')

## 3. Définitions des fonctions utilitaires

### 3A. Conversion HealSparseMap → carte HEALPix dense

La fonction `hspmap_to_healpix` convertit un objet `HealSparseMap` en tableau
HEALPix standard (RING ordering) via `hsp.generate_healpix_map(nside)`,
puis remplace les pixels hors couverture (valeur sentinelle) par `hp.UNSEEN`
pour que `healpy` les masque proprement.

In [ ]:
def hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY):
    """
    Convertit un HealSparseMap en tableau HEALPix dense (RING, nside donné).

    Parameters
    ----------
    hsp : healsparse.HealSparseMap
    nside : int
        Résolution de la carte HEALPix de sortie.

    Returns
    -------
    hpmap : np.ndarray, shape=(12*nside**2,)
        Carte HEALPix avec hp.UNSEEN aux pixels hors couverture.
    """
    hpmap = hsp.generate_healpix_map(nside=nside)
    # Remplacer la valeur sentinelle par hp.UNSEEN
    sentinel = hsp.sentinel
    hpmap[hpmap == sentinel] = hp.UNSEEN
    return hpmap


print('hspmap_to_healpix définie.')

### 3B. Normalisation couleur adaptative

Identique à la version du notebook de référence, mais réutilisée ici pour
les affichages `healpy` (vmin/vmax passés à `hp.mollview`).

In [ ]:
# Ensemble des cartes à valeurs signées (nécessitent SymLogNorm)
SYMLOG_MAPS = {
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
}


def compute_vmin_vmax(hpmap, p_low=1, p_high=99):
    """
    Calcule les percentiles p_low et p_high sur les pixels valides (≠ hp.UNSEEN).

    Returns
    -------
    vmin, vmax : float
    """
    valid = hpmap[hpmap != hp.UNSEEN]
    valid = valid[np.isfinite(valid)]
    if valid.size == 0:
        return None, None
    return float(np.percentile(valid, p_low)), float(np.percentile(valid, p_high))


print('compute_vmin_vmax définie.')

### 3C. Superposition du plan Galactique sur une vue healpy

La projection utilisée par healpy (Mollweide, gnomview…) n'accepte pas directement
de commandes de dessin Matplotlib standard : il faut passer par `hp.projplot` qui
convertit les coordonnées (RA, Dec) vers le plan de la figure.

In [ ]:
def galactic_plane_radec(n_points=2000):
    """
    Retourne les tableaux (ra, dec) qui tracent l'équateur Galactique (b=0) en ICRS.
    """
    gal_lon = np.linspace(0, 360, n_points)
    gal_lat = np.zeros(n_points)
    coords  = SkyCoord(
        l=gal_lon * u.deg,
        b=gal_lat * u.deg,
        frame=Galactic
    ).icrs
    return coords.ra.deg, coords.dec.deg


GP_RA, GP_DEC = galactic_plane_radec(n_points=2000)
print(f'Plan Galactique : {len(GP_RA)} points calculés.')


def overlay_galactic_plane(color='red', lw=1.2, alpha=0.8, label='Plan Galactique'):
    """
    Superpose le plan Galactique sur la figure healpy courante.

    À appeler **après** hp.mollview / hp.gnomview (même figure active).
    Utilise hp.projplot pour projeter (RA, Dec) → plan de la figure.
    """
    # healpy attend les coordonnées en (theta, phi) colatitude/longitude,
    # mais avec lonlat=True on peut passer (lon, lat) en degrés.
    # On passe (RA, Dec) avec coord='C' (équatoriale) -> healpy les projette.
    hp.projplot(
        GP_RA, GP_DEC,
        lonlat=True,
        coord=['C'],
        color=color,
        lw=lw,
        alpha=alpha,
        label=label
    )


print('overlay_galactic_plane définie.')

### 3D. Fonction principale d'affichage HEALPix — `show_hpmap_healpy`

Affiche une carte HealSparse en utilisant `hp.mollview` pour la projection Mollweide
(identique à l'approche de `read_healsparse.ipynb`), avec :
- un graticule (grille de coordonnées),
- des labels de latitude et longitude,
- une superposition optionnelle du plan Galactique.

In [ ]:
def show_hpmap_healpy(
    hsp,
    title='',
    unit='',
    nside=NSIDE_DISPLAY,
    cmap='viridis',
    norm_mode='hist',
    use_log=False,
    linthresh=1.0,
    map_name='',
    show_gp=True,
    lon_0=0,
    figsize=(12, 6),
):
    """
    Affiche un HealSparseMap avec hp.mollview.

    Parameters
    ----------
    hsp : healsparse.HealSparseMap
    title : str
        Titre de la figure.
    unit : str
        Étiquette de la barre de couleur.
    nside : int
        Résolution HEALPix pour la conversion.
    cmap : str
        Colormap Matplotlib.
    norm_mode : str
        'hist'   → normalisation par histogramme (healpy built-in),
        'log'    → échelle logarithmique (vmin/vmax sur percentiles),
        'symlog' → SymLogNorm,
        'linear' → linéaire (vmin/vmax sur percentiles).
    use_log : bool
        Raccourci : si True, choisit 'log' ou 'symlog' selon map_name.
    linthresh : float
        Seuil linéaire pour SymLogNorm.
    map_name : str
        Nom du dataset type (pour décider log/symlog auto).
    show_gp : bool
        Si True, superpose le plan Galactique.
    lon_0 : float
        Longitude centrale de la projection Mollweide (degrés).
    figsize : tuple
        Taille de la figure.
    """
    hpmap = hspmap_to_healpix(hsp, nside=nside)
    vmin, vmax = compute_vmin_vmax(hpmap)

    # Choisir le mode de normalisation
    if use_log:
        norm_mode = 'symlog' if (map_name in SYMLOG_MAPS) else 'log'

    if norm_mode == 'hist':
        norm_kw = dict(norm='hist')
    elif norm_mode == 'log' and vmin is not None and vmin > 0:
        hpmap = np.log10(np.where(hpmap > 0, hpmap, np.nan))
        hpmap[~np.isfinite(hpmap)] = hp.UNSEEN
        vmin2, vmax2 = compute_vmin_vmax(hpmap)
        norm_kw = dict(min=vmin2, max=vmax2)
        unit = f'log₁₀({unit})' if unit else 'log₁₀(value)'
    elif norm_mode == 'symlog':
        norm_kw = dict(min=vmin, max=vmax)
    else:  # linear
        norm_kw = dict(min=vmin, max=vmax)

    plt.figure(figsize=figsize)
    hp.mollview(
        hpmap,
        title=title,
        cmap=cmap,
        unit=unit,
        rot=(lon_0, 0, 0),
        coord=['C'],
        badcolor='lightgrey',   # ← couleur des pixels hp.UNSEEN / NaN
        bgcolor='white',         # ← couleur du fond hors de l'ellipse
        hold=False,
        **norm_kw
    )

    # Graticule
    hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)

    # Labels latitude
    for lat in [-60, -30, 0, 30, 60]:
        hp.projtext(
            lon_0 + 2, lat,
            f'{lat:+d}°',
            lonlat=True, coord='C',
            color='black', fontsize=8
        )

    # Labels longitude
    for lon in range(0, 360, 60):
        hp.projtext(
            lon, 0,
            f'{lon}°',
            lonlat=True, coord='C',
            color='black', fontsize=8
        )

    # Plan Galactique
    if show_gp:
        overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)

    plt.tight_layout()
    plt.show()


print('show_hpmap_healpy définie.')

### 3E. Affichage en grille 2×3 bandes — `display_hpmap_6bands_healpy`

Produit un panneau de 6 sous-figures (une par bande) utilisant `hp.mollview`
avec une normalisation commune calculée sur la première bande disponible.

In [ ]:
def display_hpmap_6bands_healpy(
    map_name,
    butler_obj,
    collection=None,
    bands=BANDS,
    skymap=SKYMAP_NAME,
    nside=NSIDE_DISPLAY,
    use_log=True,
    linthresh=1.0,
    cmap='viridis',
    figsize=(20, 10),
):
    """
    Affiche la carte `map_name` pour les 6 bandes dans une grille 2x3
    en utilisant hp.mollview (approche healpy native).

    La normalisation couleur est dérivée du 1er-99e percentile de la
    première bande disponible, puis appliquée à toutes les bandes.
    """
    if collection is None:
        collection = COLLECTION

    short = (
        map_name
        .replace('deepCoadd_', '')
        .replace('_consolidated', '')
    )
    is_symlog = map_name in SYMLOG_MAPS

    # ── Récupération des cartes ──────────────────────────────────────────────
    hpmaps = {}
    for band in bands:
        try:
            hsp = butler_obj.get(
                map_name,
                dataId={'band': band, 'skymap': skymap},
                collections=collection
            )
            hpmaps[band] = hspmap_to_healpix(hsp, nside=nside)
        except Exception as e:
            print(f'  [{band}] non disponible : {type(e).__name__}')

    if not hpmaps:
        print(f'Aucune carte disponible pour {map_name}.')
        return

    # ── Normalisation commune ────────────────────────────────────────────────
    first_hpmap = next(iter(hpmaps.values()))
    vmin, vmax = compute_vmin_vmax(first_hpmap)

    if use_log and vmin is not None:
        if is_symlog:
            norm_kw = dict(min=vmin, max=vmax)
        elif vmin > 0:
            norm_kw = dict(norm='hist')
        else:
            norm_kw = dict(min=vmin, max=vmax)
    else:
        norm_kw = dict(min=vmin, max=vmax)

    # ── Figure 2x3 ──────────────────────────────────────────────────────────
    nrows, ncols = 2, 3
    fig = plt.figure(figsize=figsize)
    fig.suptitle(
        f'Survey Property Map — {short}\n{collection}',
        fontsize=13, fontweight='bold'
    )

    for idx, band in enumerate(bands):
        if band not in hpmaps:
            continue

        # hp.mollview dessine dans la figure courante via sub=(nrows, ncols, idx+1)
        hp.mollview(
            hpmaps[band],
            title=f'band = {band}',
            cmap=cmap,
            coord=['C'],
            sub=(nrows, ncols, idx + 1),
            badcolor='lightgrey',   # ← couleur des pixels hp.UNSEEN / NaN
            bgcolor='white',         # ← couleur du fond hors de l'ellipse
            **norm_kw
        )
        hp.graticule(dpar=30, dmer=60, color='white', alpha=0.4, lw=0.4)

        # Plan Galactique
        overlay_galactic_plane(color='red', lw=0.8, alpha=0.6)

    plt.show()


print('display_hpmap_6bands_healpy définie.')

### 3F. Fonction de fallback PNG — `display_plots_6bands`

In [ ]:
def get_plot_path(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION):
    try:
        uri = butler.getURI(
            dataset_type,
            dataId={'band': band, 'skymap': skymap},
            collections=collection
        )
        return uri.path
    except Exception as e:
        print(f'  Non trouvé ({band}): {type(e).__name__}: {e}')
        return None


def display_plots_6bands(plot_name, df_plots, bands=BANDS, figsize=(20, 12)):
    """
    Affiche les images PNG SurveyWidePropertyMapPlot pour les 6 bandes
    dans une grille 2x3.
    """
    df_sub = df_plots[(df_plots['dataset_type'] == plot_name) & df_plots['ok']]
    if df_sub.empty:
        print(f'Aucune image PNG trouvée pour : {plot_name}')
        return

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, layout='constrained')
    axes_flat = axes.flatten()

    short = (
        plot_name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )

    for idx, band in enumerate(bands):
        ax  = axes_flat[idx]
        row = df_sub[df_sub['band'] == band]
        if row.empty:
            ax.set_visible(False)
            continue
        img = mpimg.imread(row.iloc[0]['path'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'band = {band}', fontsize=12, fontweight='bold')

    for idx in range(len(bands), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle(
        f'Survey Property Map Plots — {short}\n'
        f'(images PNG — fallback)\n{COLLECTION}',
        fontsize=12, fontweight='bold'
    )
    plt.show()


print('Fonctions fallback PNG définies.')

## 4. Initialisation du Butler

In [ ]:
butler   = Butler(REPO)
registry = butler.registry
print(f'Butler instancié sur le dépôt : {REPO}')

In [ ]:
try:
    skymap_obj = butler.get('skyMap', skymap=SKYMAP_NAME, collections=COLLECTION)
    print(f'Skymap {SKYMAP_NAME} chargée : {type(skymap_obj)}')
except Exception as e:
    print(f'Skymap non disponible : {type(e).__name__}: {e}')

## 5. Explorer les collections disponibles dans `dp2_prep`

In [ ]:
all_collections = sorted(registry.queryCollections())
print(f'Nombre total de collections : {len(all_collections)}')

In [ ]:
keywords = ['survey', 'prop', 'map', 'consolidated', 'spm', 'dp2-prep']
spm_candidates = [
    c for c in all_collections
    if any(kw.lower() in c.lower() for kw in keywords)
]
print(f'Collections candidates ({len(spm_candidates)}) :')
for c in spm_candidates:
    print(' ', c)

## 6. Recherche des dataset types de Survey Property Maps

In [ ]:
patterns = [
    '*consolidated_map*',
    '*survey_property*',
    '*property_map*',
    '*healsparse*',
    '*healSparse*',
]

found_types = set()
for pattern in patterns:
    try:
        for dtype in registry.queryDatasetTypes(expression=pattern):
            found_types.add(dtype.name)
    except Exception as e:
        print(f"Pattern '{pattern}': {e}")

print(f'\nDataset types candidats trouvés ({len(found_types)}) :')
for name in sorted(found_types):
    print(' ', name)

## 7. Séparation objets HealSparseMap / images PNG

In [ ]:
type_spmap  = []
type_spplot = []
for ftype in found_types:
    if 'PropertyMapPlot' in ftype:
        type_spplot.append(ftype)
    elif 'deepCoadd_' in ftype and '_map_' in ftype:
        type_spmap.append(ftype)

print(f'{len(type_spmap)} types HealSparseMap, {len(type_spplot)} types plot PNG.')
print('\nHealSparseMap :')
for t in sorted(type_spmap):
    print(' ', t)

## 8. Disponibilité des HealSparse maps dans la collection cible

In [ ]:
HPMAP_DATASET_TYPES = [
    'deepCoadd_dcr_ddec_consolidated_map_weighted_mean',
    'deepCoadd_dcr_dra_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e1_consolidated_map_weighted_mean',
    'deepCoadd_dcr_e2_consolidated_map_weighted_mean',
    'deepCoadd_epoch_consolidated_map_max',
    'deepCoadd_epoch_consolidated_map_mean',
    'deepCoadd_epoch_consolidated_map_min',
    'deepCoadd_exposure_time_consolidated_map_sum',
    'deepCoadd_psf_e1_consolidated_map_weighted_mean',
    'deepCoadd_psf_e2_consolidated_map_weighted_mean',
    'deepCoadd_psf_maglim_consolidated_map_weighted_mean',
    'deepCoadd_psf_size_consolidated_map_weighted_mean',
    'deepCoadd_sky_background_consolidated_map_weighted_mean',
    'deepCoadd_sky_noise_consolidated_map_weighted_mean',
]

PLOT_DATASET_TYPES = [
    'propertyMapSurvey_deepCoadd_dcr_ddec_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_dra_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_max_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_min_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_exposure_time_consolidated_map_sum_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_size_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_background_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_noise_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
]

print(f'Vérification de la disponibilité dans : {COLLECTION}')
rows_hpmap = []
for map_name in HPMAP_DATASET_TYPES:
    short = map_name.replace('deepCoadd_', '').replace('_consolidated', '')
    for band in BANDS:
        ok = False
        try:
            hsp_test = butler.get(
                map_name,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            ok = True
        except Exception:
            pass
        rows_hpmap.append({'dataset_type': map_name, 'short': short, 'band': band, 'ok': ok})

df_hpmap = pd.DataFrame(rows_hpmap)
HPMAPS_AVAILABLE = df_hpmap['ok'].any()
print(f'HealSparse maps disponibles : {HPMAPS_AVAILABLE}')
print(df_hpmap.groupby('dataset_type')['ok'].sum())

In [ ]:
# Disponibilité des images PNG
rows_plot = []
for plot_name in PLOT_DATASET_TYPES:
    short = (
        plot_name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    for band in BANDS:
        path  = None
        ok    = False
        try:
            uri  = butler.getURI(
                plot_name,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            path = uri.path
            ok   = True
        except Exception:
            pass
        rows_plot.append({
            'dataset_type': plot_name,
            'map': short,
            'band': band,
            'path': path,
            'ok': ok
        })

df = pd.DataFrame(rows_plot)
PLOTS_AVAILABLE   = df['ok'].any()
USE_PLOT_FALLBACK = (not HPMAPS_AVAILABLE) and PLOTS_AVAILABLE
print(f'PNG plots disponibles    : {PLOTS_AVAILABLE}')
print(f'Fallback PNG activé      : {USE_PLOT_FALLBACK}')

## 9A. Flags de configuration de l'affichage

- **`SHOW_HPMAP_GRID`** : Affiche une grille 2×3 de cartes HEALPix (healpy) pour chaque map type.
- **`SHOW_PLOTS_IF_NO_HPMAP`** : Repli sur les images PNG si les hpmaps sont absentes.
- **`USE_LOG_NORM`** : Applique une échelle log (utile quand les DDFs dominent).
- **`LOG_LINTHRESH`** : Seuil linéaire pour SymLogNorm.
- **`NSIDE_DISPLAY`** : Résolution HEALPix (défaut : 64).

In [ ]:
# ── Flags d'affichage ───────────────────────────────────────────────────────
SHOW_HPMAP_GRID        = True
SHOW_PLOTS_IF_NO_HPMAP = True

MAP_TO_DISPLAY  = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'
PLOT_TO_DISPLAY = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'

# Échelle logarithmique
USE_LOG_NORM  = True
LOG_LINTHRESH = 1.0

# Résolution de la carte HEALPix affichée
NSIDE_DISPLAY = 32   # modifier à 128 pour une résolution plus fine (plus lent)

print(f'NSIDE_DISPLAY    = {NSIDE_DISPLAY}')
print(f'USE_LOG_NORM     = {USE_LOG_NORM}')
print(f'HPMAPS_AVAILABLE = {HPMAPS_AVAILABLE}')
print(f'PLOTS_AVAILABLE  = {PLOTS_AVAILABLE}')

## 9B. Fonction helper — plan Galactique

(déjà définie en §3C — vérification ici)

In [ ]:
print(f'GP_RA  : {len(GP_RA)} points, RA ∈ [{GP_RA.min():.1f}, {GP_RA.max():.1f}]°')
print(f'GP_DEC : {len(GP_DEC)} points, Dec ∈ [{GP_DEC.min():.1f}, {GP_DEC.max():.1f}]°')

## 9C. Affichage HEALPix — grille 2×3 bandes (healpy)

### Approche `read_healsparse.ipynb` appliquée à l'ensemble des cartes disponibles

Pour chaque type de carte disponible, on :
1. Récupère la `HealSparseMap` via le Butler,
2. Convertit en carte HEALPix dense (`hsp.generate_healpix_map`),
3. Affiche avec `hp.mollview` (projection Mollweide) + graticule + plan Galactique.

La normalisation est calculée sur le percentile 1–99 de la première bande disponible,
puis appliquée uniformément aux 6 bandes pour permettre la comparaison inter-bandes.

#### 9C-1. Exemple sur une seule carte et une seule bande (carte psf_maglim, bande r)

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        hsp_demo = butler.get(
            MAP_TO_DISPLAY,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=COLLECTION
        )
        print(f'Type : {type(hsp_demo)}')
        print(f'Pixels valides : {hsp_demo.n_valid}')
        print(f'nside_sparse   : {hsp_demo.nside_sparse}')
        print(f'nside_coverage : {hsp_demo.nside_coverage}')

        show_hpmap_healpy(
            hsp_demo,
            title=f'{MAP_TO_DISPLAY}\nband={BAND_REF}  (nside={NSIDE_DISPLAY})',
            unit='mag AB',
            nside=NSIDE_DISPLAY,
            cmap='viridis',
            norm_mode='hist',
            show_gp=True,
        )
    except Exception as e:
        print(f'Erreur : {type(e).__name__}: {e}')
else:
    print('HealSparse maps non disponibles — section ignorée.')

#### 9C-2. Grille 2×3 bandes pour la carte de référence (psf_maglim)

In [ ]:
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    print(f'Affichage de la grille 6 bandes pour : {MAP_TO_DISPLAY}')
    display_hpmap_6bands_healpy(
        MAP_TO_DISPLAY,
        butler,
        nside=NSIDE_DISPLAY,
        use_log=USE_LOG_NORM,
        linthresh=LOG_LINTHRESH,
        cmap='viridis',
    )
else:
    print('Affichage HEALPix ignoré (SHOW_HPMAP_GRID=False ou maps non disponibles).')

#### 9C-3. Boucle sur tous les types de cartes disponibles

In [ ]:
if SHOW_HPMAP_GRID and HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    available_hpmap_names = (
        df_hpmap[df_hpmap['ok']]['dataset_type'].unique().tolist()
    )
    print(f'Affichage de {len(available_hpmap_names)} types de cartes...')
    for map_name in available_hpmap_names:
        print(f'\n--- {map_name} ---')
        display_hpmap_6bands_healpy(
            map_name,
            butler,
            nside=NSIDE_DISPLAY,
            use_log=USE_LOG_NORM,
            linthresh=LOG_LINTHRESH,
            cmap='viridis',
        )
else:
    print('Boucle healpy ignorée.')

## 9D. Fallback PNG — grille 2×3 bandes

Activé uniquement si `USE_PLOT_FALLBACK = True`
(aucune HealSparseMap disponible mais des images PNG existent).

In [ ]:
if USE_PLOT_FALLBACK:
    for plot_name in PLOT_DATASET_TYPES:
        df_check = df[(df['dataset_type'] == plot_name) & df['ok']]
        if df_check.empty:
            continue
        short = (
            plot_name
            .replace('propertyMapSurvey_deepCoadd_', '')
            .replace('_SurveyWidePropertyMapPlot', '')
        )
        print(f'\n--- {short} ---')
        display_plots_6bands(plot_name, df)
else:
    print('Fallback PNG ignoré (HealSparse maps disponibles ou fallback désactivé).')

## 10. Valeur de la carte au champ de référence

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        hspmap_ref = butler.get(
            MAP_TO_DISPLAY,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=COLLECTION
        )
        val_center = hspmap_ref.get_values_pos(RA_CEN, DEC_CEN)
        print(f"Carte '{MAP_TO_DISPLAY}' (bande {BAND_REF}) au centre de {FIELD}")
        print(f'  RA={RA_CEN}, Dec={DEC_CEN}  ->  {val_center}')
        val_north = hspmap_ref.get_values_pos(180.0, +60.0)
        print(f'  Valeur sentinelle (hors couverture, RA=180, Dec=+60) -> {val_north}')
    except Exception as e:
        print(f'Erreur : {type(e).__name__}: {e}')
else:
    print('Carte non disponible — section ignorée.')

## 11. Statistiques dans une région autour du champ de référence

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        span_dec = 2.0
        span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))

        ra  = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  250)
        dec = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 250)
        x, y = np.meshgrid(ra, dec)

        values   = hspmap_ref.get_values_pos(x, y)
        sentinel = hspmap_ref.get_values_pos(180.0, +60.0)
        values   = values.astype(float)
        values[values == sentinel] = np.nan

        print(f'Région {FIELD} — {MAP_TO_DISPLAY} (bande {BAND_REF})')
        print(f'  min    = {np.nanmin(values):.4f}')
        print(f'  max    = {np.nanmax(values):.4f}')
        print(f'  mean   = {np.nanmean(values):.4f}')
        print(f'  median = {np.nanmedian(values):.4f}')
        print(f'  std    = {np.nanstd(values):.4f}')
    except Exception as e:
        print(f'Erreur : {type(e).__name__}: {e}')
else:
    print('Carte non disponible — section ignorée.')

## 12. Visualisation locale (pcolormesh)

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    try:
        fig, ax = plt.subplots(figsize=(7, 6))
        pcm = ax.pcolormesh(x, y, values, cmap='viridis')
        fig.colorbar(pcm, ax=ax, label=f"{MAP_TO_DISPLAY.split('_')[-2]} ({BAND_REF}-band)")
        ax.set_xlabel('Right Ascension (deg)')
        ax.set_ylabel('Declination (deg)')
        ax.set_title(f'{MAP_TO_DISPLAY}\n{BAND_REF}-band — {FIELD}', fontsize=11)
        ax.invert_xaxis()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Erreur : {type(e).__name__}: {e}')
else:
    print(f'Carte non disponible pour {FIELD} — visualisation ignorée.')

## 13. Vue tout-ciel healpy — boucle sur toutes les cartes disponibles

Pour chaque carte disponible et pour chaque bande, on produit une vue Mollweide
individuelle avec `hp.mollview` (approche directe de `read_healsparse.ipynb`).
On ajoute en superposition le plan Galactique.

### 13A. Vue individuelle healpy pour un seul type de carte, toutes bandes

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    map_name_single = MAP_TO_DISPLAY
    short_single    = map_name_single.replace('deepCoadd_', '').replace('_consolidated', '')

    for band in BANDS:
        try:
            hsp = butler.get(
                map_name_single,
                dataId={'band': band, 'skymap': SKYMAP_NAME},
                collections=COLLECTION
            )
            hpmap = hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY)

            plt.figure(figsize=(12, 6))
            hp.mollview(
                hpmap,
                title=f'{short_single}  —  band = {band}  (nside={NSIDE_DISPLAY})',
                cmap='viridis',
                unit='mag AB',
                norm='hist',
                coord=['C'],
                badcolor='lightgrey',   # ← couleur des pixels hp.UNSEEN / NaN
                bgcolor='white',         # ← couleur du fond hors de l'ellipse
            )
            hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)
            # Labels latitude
            for lat in [-60, -30, 0, 30, 60]:
                hp.projtext(2, lat, f'{lat:+d}°', lonlat=True, coord='C',
                            color='black', fontsize=8)
            # Labels longitude
            for lon in range(0, 360, 60):
                hp.projtext(lon, 0, f'{lon}°', lonlat=True, coord='C',
                            color='black', fontsize=8)
            # Plan Galactique
            overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f'  [{band}] non disponible : {type(e).__name__}')
else:
    print('HealSparse maps non disponibles — section ignorée.')

### 13B. Boucle sur tous les types de cartes HEALPix disponibles — vues individuelles

In [ ]:
if HPMAPS_AVAILABLE and HAS_HEALSPARSE:
    available_types = df_hpmap[df_hpmap['ok']]['dataset_type'].unique().tolist()
    print(f'{len(available_types)} types de cartes à afficher.')

    for map_name in available_types:
        short  = map_name.replace('deepCoadd_', '').replace('_consolidated', '')
        is_sym = map_name in SYMLOG_MAPS
        print(f'\n=== {short} ===')

        available_bands = (
            df_hpmap[(df_hpmap['dataset_type'] == map_name) & df_hpmap['ok']]
            ['band'].tolist()
        )

        for band in available_bands:
            try:
                hsp   = butler.get(
                    map_name,
                    dataId={'band': band, 'skymap': SKYMAP_NAME},
                    collections=COLLECTION
                )
                hpmap = hspmap_to_healpix(hsp, nside=NSIDE_DISPLAY)
                vmin, vmax = compute_vmin_vmax(hpmap)

                norm_kw = dict(norm='hist') if not is_sym else dict(min=vmin, max=vmax)

                plt.figure(figsize=(12, 6))
                hp.mollview(
                    hpmap,
                    title=f'{short}  —  band = {band}',
                    cmap='viridis',
                    coord=['C'],
                    badcolor='lightgrey',   # ← couleur des pixels hp.UNSEEN / NaN
                    bgcolor='white',         # ← couleur du fond hors de l'ellipse
                    **norm_kw
                )
                hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)
                for lat in [-60, -30, 0, 30, 60]:
                    hp.projtext(2, lat, f'{lat:+d}°', lonlat=True, coord='C',
                                color='black', fontsize=8)
                for lon in range(0, 360, 60):
                    hp.projtext(lon, 0, f'{lon}°', lonlat=True, coord='C',
                                color='black', fontsize=8)
                overlay_galactic_plane(color='red', lw=1.2, alpha=0.8)
                plt.tight_layout()

                # Sauvegarde optionnelle
                fig_path = os.path.join(
                    DIR_FIGS,
                    f'healpix_{short}_{band}_nside{NSIDE_DISPLAY}.png'
                )
                plt.savefig(fig_path, dpi=120, bbox_inches='tight')
                plt.show()

            except Exception as e:
                print(f'  [{band}] erreur : {type(e).__name__}: {e}')
else:
    print('Boucle sur toutes les cartes ignorée.')

## 14. Notes et prochaines étapes

### Différences avec le notebook de référence

| Aspect | `2026-03-11` (référence) | `2026-03-12` (ce notebook) |
|---|---|---|
| Projection tout-ciel | `skyproj.McBrydeSkyproj` | `hp.mollview` (healpy natif) |
| Conversion HealSparse | `skyproj.draw_hspmap` | `hsp.generate_healpix_map` |
| Graticule | `skyproj` built-in | `hp.graticule` + `hp.projtext` |
| Plan Galactique | `skyproj` overlay | `hp.projplot` via `overlay_galactic_plane` |
| Normalisation | percentile sur `get_values_flat` | percentile sur `hpmap[hpmap != hp.UNSEEN]` |

### Prochaines étapes

- Utiliser `hp.gnomview` à la place de `hp.mollview` pour une vue locale haute résolution.
- Combiner `hp.projplot` avec des catalogues d'objets (sources, clusters, …).
- Comparer les valeurs pixel-à-pixel entre bandes via `hp.ud_grade`.
- Exporter les cartes HEALPix au format FITS avec `hp.write_map`.
- Étendre la comparaison DP1 vs DP2 sur les mêmes nside et champs.